# Project 1: Analyzing NYC Restaurant Health Scores

Welcome to my first data project! In this notebook, we are going to explore how well restaurants in New York City perform on their official health inspections. 

To do this, I am using the [DOHMH New York City Restaurant Inspection Results dataset](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j/about_data) from NYC Open Data. 

I downloaded a sample of 1,000 recent inspections. The dataset includes a numeric `score` column—where a lower score actually means a cleaner restaurant! Let's load the data using Pandas to see what we are working with.

In [ ]:
import pandas as pd

# 1. Read in the data
df = pd.read_csv("restaurant_scores.csv")

# Clean the data: drop any rows that don't have a health score yet
df = df.dropna(subset=["score"])

# 2. Calculate Mean, Median, and Mode using Pandas
mean_score = df["score"].mean()
median_score = df["score"].median()
# .mode() returns a series, so we use .iloc[0] to extract the actual numeric value
mode_score = df["score"].mode().iloc[0]

print(f"The average (mean) health violation score is: {mean_score:.2f}")
print(f"The median health violation score is: {median_score}")
print(f"The most common (mode) health violation score is: {mode_score}")

The average (mean) health violation score is: 18.08
The median health violation score is: 13.0
The most common (mode) health violation score is: 0.0


### Doing the Math "The Hard Way"

Pandas makes data analysis incredibly easy, but it's important to understand the math happening behind the scenes. What if we didn't have Pandas or external libraries? 

Let's calculate the exact same metrics—mean, median, and mode—using only standard Python. We will open the CSV file manually, extract the scores into a list, and write our own algorithms to crunch the numbers.

In [ ]:
import csv

# 1. Read the data manually
scores = []
with open("restaurant_scores.csv", "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        # Only grab rows that actually have a score
        if row["score"]:
            scores.append(float(row["score"]))

# 2. Calculate the Mean
total_sum = sum(scores)
count = len(scores)
hard_mean = total_sum / count

# 3. Calculate the Median
scores.sort()  # Sort from lowest to highest
mid_index = count // 2

if count % 2 == 0:
    # If even number of items, average the middle two
    hard_median = (scores[mid_index - 1] + scores[mid_index]) / 2
else:
    # If odd, just grab the exact middle
    hard_median = scores[mid_index]

# 4. Calculate the Mode using a dictionary
frequency = {}
for score in scores:
    if score in frequency:
        frequency[score] += 1
    else:
        frequency[score] = 1

# Find the score with the highest count
hard_mode = max(frequency, key=frequency.get)

print(f"Hard Way Mean: {hard_mean:.2f}")
print(f"Hard Way Median: {hard_median}")
print(f"Hard Way Mode: {hard_mode}")

Hard Way Mean: 18.08
Hard Way Median: 13.0
Hard Way Mode: 0.0


### Visualizing the Grades

Numbers are great, but visuals are better. In NYC, a score of 0-13 is an 'A', 14-27 is a 'B', and 28+ is a 'C' or worse. 

Let's build a text-based histogram using standard Python to visualize the distribution of these grades across our sample. Each block symbol will represent roughly 10 restaurants.

In [ ]:
# Group the scores into NYC grade buckets
grade_a = sum(1 for s in scores if s <= 13)
grade_b = sum(1 for s in scores if 14 <= s <= 27)
grade_c = sum(1 for s in scores if s >= 28)

# Scale down the visual so it fits nicely on the screen (1 symbol = 10 restaurants)
scale = 10

print("NYC Restaurant Health Grades Distribution")
print("========================================")
print(f"Grade A (0-13)  : {'🟩' * (grade_a // scale)} ({grade_a})")
print(f"Grade B (14-27) : {'🟨' * (grade_b // scale)} ({grade_b})")
print(f"Grade C (28+)   : {'🟥' * (grade_c // scale)} ({grade_c})")
print("========================================")
print(f"Key: Each colored square represents ~{scale} restaurants.")

NYC Restaurant Health Grades Distribution
Grade A (0-13)  : 🟩🟩🟩🟩🟩🟩🟩🟩🟩 (94)
Grade B (14-27) : 🟨🟨🟨🟨 (40)
Grade C (28+)   : 🟥🟥🟥🟥 (46)
Key: Each colored square represents ~10 restaurants.


### Caveats and Data Quality

While our math is correct for the data provided, it is important to note a few caveats about this dataset:
1. **Inspections vs. Restaurants:** This dataset records individual *inspections*. Because a single restaurant can be inspected multiple times (especially if they fail initially), our mean score of 18.08 represents the average of the *inspections*, not necessarily the true average of all *restaurants*.
2. **Missing Data:** We had to drop rows where the `score` column was blank. These blanks likely represent new restaurants that have applied for a permit but haven't received their initial graded inspection yet.